In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from config import Config
from feature_engineering import create_subset, drop_features, engineer_features, h
from feature_selection import run_feature_selection_pipeline
from plotting import plot_all

import pandas as pd
%matplotlib inline

In [ ]:
test = pd.read_csv(Config.LMD_2022_DIR / "LMD-2022_NormalVsMalicious.csv")

/tmp/ipykernel_32697/2248565981.py:1: DtypeWarning: Columns (0: ParentUser, 1: State, 2: SourceUser, 3: TargetUser, 4: CreationUtcTime, 5: TargetObject, 6: Details, 7: PipeName, 8: QueryName, 9: QueryResults) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(Config.LMD_2022_DIR / "testingfile.csv")


In [14]:
test.Label.value_counts()

Label
1    110710
Name: count, dtype: int64

In [2]:
df = pd.read_csv(Config.LMD_2022_PATH, low_memory=False)

useful_cols = []
for col in df.columns:
    if not col.startswith('Unnamed'):
        useful_cols.append(col)
df = df[useful_cols].copy()

df.rename(columns={'Label': 'isMalicious'}, inplace=True)

valid_rows = (df['isMalicious'] == '1') | (df['isMalicious'] == '0')
df = df[valid_rows].copy()
df['isMalicious'] = df['isMalicious'].astype(int)

print('Rows:', len(df))
print(df['isMalicious'].value_counts())

Rows: 293548
isMalicious
1    191194
0    102354
Name: count, dtype: int64


In [3]:
raw_df = df.copy()
df = create_subset(df)


SECTION 0: STRATIFIED SUBSET
Stratified subset: 50000 rows from 293548


In [4]:
df_clean, all_drops, cols_before = drop_features(df)


SECTION 1: FEATURE REMOVAL
1a. Near-zero variance columns: 15
1b. Metadata / identifiers: 25
1c. High-cardinality leak risk: 11
1d. Redundant: 3

Features before: 92
Features after:  42
Total dropped:   50


In [5]:
X, y, groups = engineer_features(df_clean)


SECTION 2: FEATURE ENGINEERING
Done: Image -> Image_Category (6 categories)
Done: SourcePort -> SourcePort_Bin (4 bins)
Done: DestinationPort -> DestinationPort_Bin (4 bins)
Done: ImageLoaded - grouped 141 rare categories
Done: ParentImage - grouped 88 rare categories
Done: TargetImage - grouped 54 rare categories
Done: SourceImage - grouped 16 rare categories
Done: CurrentDirectory - grouped 55 rare categories
Done: QueryName - grouped 123 rare categories
Done: LogonId -> LogonId_Populated (binary)
Done: Dropped ParentProcessId (identifier)

Final X shape: (50000, 41)
Categorical columns: 38
Numeric columns:     3


In [6]:
results = run_feature_selection_pipeline(X, y, groups, output_dir='.')


SECTION 3: GROUP-BASED TRAIN/VAL SPLIT
Train samples: 36952
Val samples:   13048

SECTION 4: ONE-HOT ENCODING + VARIANCE FILTER
Encoded features: 1744
After variance filter: 1744

SECTION 5: MUTUAL INFORMATION

SECTION 6: RF IMPORTANCE RANKING

SECTION 7: AGGREGATE TO ORIGINAL FEATURES

SECTION 8: FEATURE-COUNT SWEEP
  Top-3: Val F1=0.99  Gap=-0.0513
  Top-5: Val F1=0.9995  Gap=-0.0354
  Top-6: Val F1=0.9995  Gap=-0.0354
  Top-7: Val F1=0.5941  Gap=0.3905
  Top-8: Val F1=0.5941  Gap=0.3905
  Top-9: Val F1=0.5941  Gap=0.3911
  Top-10: Val F1=0.5941  Gap=0.392
  Top-11: Val F1=0.5942  Gap=0.3961
  Top-12: Val F1=0.5942  Gap=0.3962
  Top-13: Val F1=0.5942  Gap=0.3962

SECTION 9: OPTIMAL FEATURE COUNT (PARETO)
Best Val F1: 0.9995
Pareto choice: 5 features

Selected 5 features:
  1. Image_Category
  2. GrantedAccess
  3. RuleName
  4. Signed
  5. SourceImage

Validation Report (5 features, unseen computers):
              precision    recall  f1-score   support

      Normal     0.9990    

In [7]:
print('Final', results['OPTIMAL_N'], 'features:')
for i in range(len(results['final_features'])):
    print(' ', i + 1, results['final_features'][i])

print()
print(results['sweep_df'].to_string(index=False))

Final 5 features:
  1 Image_Category
  2 GrantedAccess
  3 RuleName
  4 Signed
  5 SourceImage

 Original_Features  Encoded_Features  Train_F1  Val_F1  Val_F1_Normal  Val_F1_Attack  Overfit_Gap                                                                                                                                      Selected_Features
                 3                58    0.9387  0.9900         0.9911         0.9888      -0.0513                                                                                                                Image_Category, GrantedAccess, RuleName
                 5                75    0.9641  0.9995         0.9995         0.9994      -0.0354                                                                                           Image_Category, GrantedAccess, RuleName, Signed, SourceImage
                 6               103    0.9641  0.9995         0.9995         0.9994      -0.0354                                                            

In [8]:
plot_all(raw_df, results, output_dir='.')

Saved: ./01_class_distribution.png
Saved: ./02_event_counts.png


/home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/src/plotting.py:140: UserWarning: Glyph 131 (\x83) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/src/plotting.py:140: UserWarning: Glyph 130 (\x82) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/src/plotting.py:143: UserWarning: Glyph 131 (\x83) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches="tight")
/home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/src/plotting.py:143: UserWarning: Glyph 130 (\x82) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=300, bbox_inches="tight")


Saved: ./03_mi_encoded_features.png
Saved: ./04_original_feature_importance.png
Saved: ./05_performance_vs_features.png
Saved: ./06_overfit_gap.png
Saved: ./07_per_class_f1.png

All 7 charts saved to: .


In [ ]:
from prepare_lmd2023_test import prepare_lmd2023_test
prepare_lmd2023_test()

In [15]:
from preprocessing_pipeline import create_all_datasets
create_all_datasets()

CREATING ALL DATASETS
Selected features: ['Image_Category', 'GrantedAccess', 'RuleName', 'Signed', 'SourceImage']

Loading: /home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/data/lmd2022/LMD-2022_NormalVsMalicious.csv
  Raw rows: 293548
  Created Image_Category from Image column
  Filtered to label=0: 102354 rows
  Final: 102354 rows x 6 cols

Loading: /home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/data/lmd2022/LMD-2022_NormalVsMalicious.csv
  Raw rows: 293548
  Created Image_Category from Image column
  Final: 293548 rows x 6 cols

Loading: /home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/data/lmd2023/lmd2023_test_file.csv
  Raw rows: 1752844
  Created Image_Category from Image column
  Final: 1752844 rows x 6 cols

Loading (with sequence cols): /home/mike/Documents/Sustainable-Machine-Learning-for-Lateral-Movement-Detection/data/lmd2022/LMD-2022_NormalVsMalicious.csv
  Final: 293548 rows
